# ODI to Databricks Migration

## Converted from TARGET ODI SQL File

**Conversion Timestamp:** 2024-07-30T12:00:00Z

**Description:** This notebook contains the Databricks Spark SQL conversion of the original ODI SQL file, focused on loading department data into `workspace.hr.trg_dep`.

In [ ]:
dbutils.widgets.text("ETL_JOB_TYPE", "F", "1. ETL Job Type (F/I)")
dbutils.widgets.text("DATASOURCE_NUM_ID", "100", "2. Datasource Number ID")
dbutils.widgets.text("ETL_PROC_WID", "1", "3. ETL Process ID")
dbutils.widgets.text("ODI_SESS_NO", "12345", "4. ODI Session Number")
dbutils.widgets.text("V_ETL_LAST_EXTRACT_TIME", "1900-01-01 00:00:00", "5. Last Extract Timestamp (YYYY-MM-DD HH:mm:ss)")
dbutils.widgets.text("V_ETL_CURRENT_EXTRACT_TIME", "9999-12-31 23:59:59", "6. Current Extract Timestamp (YYYY-MM-DD HH:mm:ss)")

# ETL Parameters

In [ ]:
%sql
-- Create temporary views for ETL parameters
CREATE OR REPLACE TEMPORARY VIEW v_etl_job_type AS
SELECT '${ETL_JOB_TYPE}' AS etl_job_type;

CREATE OR REPLACE TEMPORARY VIEW v_datasource_num_id AS
SELECT CAST(${DATASOURCE_NUM_ID} AS BIGINT) AS datasource_num_id;

CREATE OR REPLACE TEMPORARY VIEW v_etl_proc_wid AS
SELECT CAST(${ETL_PROC_WID} AS BIGINT) AS etl_proc_wid;

CREATE OR REPLACE TEMPORARY VIEW v_odi_sess_no AS
SELECT '${ODI_SESS_NO}' AS odi_sess_no;

CREATE OR REPLACE TEMPORARY VIEW v_etl_last_extract_time AS
SELECT to_timestamp('${V_ETL_LAST_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_last_extract_time;

CREATE OR REPLACE TEMPORARY VIEW v_etl_current_extract_time AS
SELECT to_timestamp('${V_ETL_CURRENT_EXTRACT_TIME}', 'yyyy-MM-dd HH:mm:ss') AS etl_current_extract_time;

In [ ]:
display(spark.sql("""
  SELECT
    (SELECT etl_job_type FROM v_etl_job_type) AS ETL_JOB_TYPE,
    (SELECT datasource_num_id FROM v_datasource_num_id) AS DATASOURCE_NUM_ID,
    (SELECT etl_proc_wid FROM v_etl_proc_wid) AS ETL_PROC_WID,
    (SELECT odi_sess_no FROM v_odi_sess_no) AS ODI_SESS_NO,
    (SELECT etl_last_extract_time FROM v_etl_last_extract_time) AS ETL_LAST_EXTRACT_TIME,
    (SELECT etl_current_extract_time FROM v_etl_current_extract_time) AS ETL_CURRENT_EXTRACT_TIME
"""))

# Staging Table Processing

In [ ]:
%sql
-- SCEN_TASK_NO {10}: Placeholder for initial staging setup or checks

In [ ]:
%sql
-- SCEN_TASK_NO {20}: Placeholder for data selection into staging

In [ ]:
%sql
-- SCEN_TASK_NO {30}: Placeholder for staging data transformations

# Target Load

In [ ]:
%sql
INSERT INTO workspace.hr.trg_dep
  (
    department_id,
    department_name,
    manager_id,
    location_id
  )
SELECT
  departments.department_id,
  departments.department_name,
  departments.manager_id,
  departments.location_id
FROM
  workspace.hr.departments AS departments;

# Cleanup

In [ ]:
%sql
-- No temporary staging or flow tables were explicitly created in this simple scenario, hence no specific DROP statements. 
-- If C$ or I$ tables were used, they would be dropped here, e.g.:
-- DROP TABLE IF EXISTS workspace.hr.c_staging_table;
-- DROP TABLE IF EXISTS workspace.hr.i_flow_table;

# Validation

In [ ]:
%sql
SELECT COUNT(*) AS total_records FROM workspace.hr.trg_dep;

# Conversion Notes and Manual Actions Required

1.  **Schema and Table Names:** All schema names like `HR` have been converted to `workspace.hr` and table names to lowercase (`trg_dep`, `departments`).
2.  **Oracle Hints:** The `/*+ APPEND PARALLEL */` hint has been removed as it is not applicable in Databricks Spark SQL for Delta tables.
3.  **SCEN_TASK_NO:** The `SCEN_TASK_NO` markers were converted to comments within empty SQL cells, preserving the original sequence of tasks.
4.  **Data Types:** Without the `CREATE TABLE` DDL for `TRG_DEP` and `DEPARTMENTS`, explicit data type conversions (e.g., `NUMBER` to `BIGINT`/`DECIMAL`, `VARCHAR2` to `STRING`) were not applied directly in this INSERT statement. It's assumed the target table `workspace.hr.trg_dep` has already been created with compatible Spark SQL data types.
5.  **Error Handling/Logging:** The original ODI file did not contain explicit error table (`E$_*`) logic or `SNP_CHECK_TAB` updates, so these were not generated. If present in a more complex ODI flow, they would be included.
6.  **ETL Parameters:** Standard ETL parameters (Job Type, Datasource ID, Process WID, Session Number, Last/Current Extract Timestamps) have been set up as Databricks widgets and temporary views for common use in larger ETL flows, even though they are not directly referenced in this simple `INSERT` statement.
7.  **Semicolons:** A semicolon has been added at the end of the SQL statement as required for Spark SQL in Databricks.